#### Author's Information:
        Author : Ali Rizvi.
        Stu ID : 2410057
        Module : HEP502 of MSc Computer Sciences with Artificial Intelligence.
        Title  : Comparative Evaluation of MLP and CNN on Fashion-MNIST.
        Purpose: Final Assessment
---


| #| ## of Notebook | Title of Notebook |
|--|---|---|
| 1st| 00 | Data Loading and Sanity Checks |
| 2| 01 | Baseline MLP |
| 3| 02 | Baseline CNN |
| 4| 03 | Regularisation Experiments |
| **5**| **04** | **Depth and Capacity Variance** |
| 6| 05 | Robustness Testing |
| 7| 06 | Consolidated Analysis |

**This is Notebook 04,** 5th of this 7-notebook series. The following are the goals of this notebook:
## Goals:
Experiment and Explore:
- MLP with a single, two, and three hidden layers.
- CNN with 2 Conv blocks, then a wider CNN with more filters, and finally a deeper CNN with 3 Conv blocks.
- Measure accuracies such as validation and test, and other metrics such as generalization gap, parameter count, and etc.

**This notebook contains experimentations with Depth and Capacity variance.**

#### Importing libraries

In [1]:
import json
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

#### Loading data, setting up directories and seed.

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = Path("outputs")
FIG_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"
MODEL_DIR = OUTPUT_DIR / "models"

directories = [OUTPUT_DIR, FIG_DIR, LOG_DIR, MODEL_DIR]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    
print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")

Outputs will be saved to: C:\Users\ALI\PG\HEP502_work\alirizvi_2410057_hep502\code_mlp_vs_cnn\outputs


In [3]:
data_path = OUTPUT_DIR / "fashion_mnist_prepared_00.npz"
data = np.load(data_path)

x_train = data["x_train"]
y_train = data["y_train"]
x_val = data["x_val"]
y_val = data["y_val"]
x_test = data["x_test"]
y_test = data["y_test"]

# Prepare shapes
# MLP
x_train_mlp = x_train.reshape((x_train.shape[0], -1))
x_val_mlp = x_val.reshape((x_val.shape[0], -1))
x_test_mlp = x_test.reshape((x_test.shape[0], -1))
# CNN
x_train_cnn = x_train[:, :, :, np.newaxis]
x_val_cnn = x_val[:, :, :, np.newaxis]
x_test_cnn = x_test[:, :, :, np.newaxis]

#### Flexible Config

In [4]:
@dataclass(frozen=True)
class DepthConfig:
    # Config focused purely on architecturial depth/width variations
    arch: str  # "mlp" or "cnn"
    mlp_layers: int = 1
    mlp_units: int = 128
    cnn_filters: tuple = (32, 64)  # one entry per conv block

    def run_name(self):
        # Create readable identifier
        if self.arch == "mlp":
            return f"mlp_depth{self.mlp_layers}"
        else:
            return f"cnn_filters{'-'.join(map(str, self.cnn_filters))}"

### Varying Depths

#### MLP

In [5]:
def build_mlp_depth(config: DepthConfig, input_shape=(784,), num_classes=10):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    # Stack Dense layers according to the requested depth
    for _ in range(config.mlp_layers):
        x = layers.Dense(config.mlp_units, activation="relu")(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name=config.run_name())

#### CNN

In [6]:
def build_cnn_depth(config: DepthConfig, input_shape=(28, 28, 1), num_classes=10):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    
    # Create one conv + pooling block for each entry in cnn_filters
    for filters in config.cnn_filters:
        x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
        x = layers.MaxPooling2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name=config.run_name())

#### Training

In [7]:
def train_depth_model(config: DepthConfig):
    tf.keras.backend.clear_session()  # avoid graph/weight buildup when training lots of models in a loop
    
    # Choose model and matching input representation
    if config.arch == "mlp":
        model = build_mlp_depth(config)
        x_tr, x_va, x_te = x_train_mlp, x_val_mlp, x_test_mlp
    else:
        model = build_cnn_depth(config)
        x_tr, x_va, x_te = x_train_cnn, x_val_cnn, x_test_cnn

    # Keep training setup consistent across depth experiments
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    start = time.time()

    history = model.fit(
        x_tr, y_train,
        validation_data=(x_va, y_val),  # lets us see if models overfit
        epochs=20,
        batch_size=128,
        verbose=0
    )

    train_time = time.time() - start

    test_loss, test_acc = model.evaluate(x_te, y_test, verbose=0)

    num_params = model.count_params()
    train_acc = history.history["accuracy"][-1]
    val_acc = history.history["val_accuracy"][-1]

    # Return a compact record
    return {
        "run_name": config.run_name(),
        "arch": config.arch,
        "test_acc": float(test_acc),
        "val_acc": float(val_acc),
        "train_acc": float(train_acc),
        "gap": float(train_acc - val_acc),  # simple overfitting indicator
        "params": int(num_params),
        "time_sec": float(train_time),
        "acc_per_10k_params": float(test_acc / (num_params / 10000))
    }

#### Defining Depth Experiments

In [8]:
depth_experiments = [

    # MLP depth
    DepthConfig(arch="mlp", mlp_layers=1),
    DepthConfig(arch="mlp", mlp_layers=2),
    DepthConfig(arch="mlp", mlp_layers=3),

    # CNN width variations
    DepthConfig(arch="cnn", cnn_filters=(32, 64)),
    DepthConfig(arch="cnn", cnn_filters=(64, 128)),
    DepthConfig(arch="cnn", cnn_filters=(32, 64, 128)),
]

#### Running Experiments

In [9]:
depth_results = []

for cfg in depth_experiments:
    print("Running:", cfg.run_name())
    result = train_depth_model(cfg)
    depth_results.append(result)
    print(
        f"-> test={result['test_acc']:.4f} | "
        f"gap={result['gap']:.4f} | "
        f"params={result['params']} | "
        f"time={result['time_sec']:.1f}s"
    )

# Sort the results
depth_results = sorted(depth_results, key=lambda x: x["test_acc"], reverse=True)

# Display the results
for r in depth_results:
    print(
        f"{r['run_name']:<20} | "
        f"test={r['test_acc']:.4f} | "
        f"gap={r['gap']:.4f} | "
        f"params={r['params']} | "
        f"time={r['time_sec']:.1f}s"
    )

Running: mlp_depth1

-> test=0.8732 | gap=0.0447 | params=101770 | time=98.8s
Running: mlp_depth2
-> test=0.8685 | gap=0.0602 | params=118282 | time=99.8s
Running: mlp_depth3
-> test=0.8704 | gap=0.0575 | params=134794 | time=95.5s
Running: cnn_filters32-64
-> test=0.9099 | gap=0.0694 | params=421642 | time=734.4s
Running: cnn_filters64-128
-> test=0.9162 | gap=0.0722 | params=878730 | time=1524.0s
Running: cnn_filters32-64-128
-> test=0.9123 | gap=0.0664 | params=241546 | time=1059.6s
cnn_filters64-128    | test=0.9162 | gap=0.0722 | params=878730 | time=1524.0s
cnn_filters32-64-128 | test=0.9123 | gap=0.0664 | params=241546 | time=1059.6s
cnn_filters32-64     | test=0.9099 | gap=0.0694 | params=421642 | time=734.4s
mlp_depth1           | test=0.8732 | gap=0.0447 | params=101770 | time=98.8s
mlp_depth3           | test=0.8704 | gap=0.0575 | params=134794 | time=95.5s
mlp_depth2           | test=0.8685 | gap=0.0602 | params=118282 | time=99.8s
